### Geographic pattern analysis for voter turnouts in Salem

In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly as px
import folium
from folium.plugins import MarkerCluster
from sklearn.cluster import KMeans
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.cm as cm
import matplotlib.colors as colors


#### Exploratory data analysis

In [35]:
df = pd.read_csv("cleaned_data.csv")

/var/folders/n5/_35mdxvd5tg02wjzpxdfqp0m0000gn/T/ipykernel_9203/1989269044.py:1: DtypeWarning: Columns (12,23,24,28,30,39,45,46,50,52,73,161,162,163,164,165,166,167,169,170,171,172,173,174,175,178,179,182,184,187,194,195,197,206,214,226,232,233,244,260,267,269,489,490,511) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_data.csv")


In [31]:
df.head()

,Unnamed: 0,SEQUENCE,LALVOTERID,Voters_StateVoterID,VoterTelephones_Phone10,VoterTelephones_FullPhone,VoterTelephones_TelAreaCode,VoterTelephones_TelCellFlag,VoterTelephones_TelConfidenceCode,View_Ops_CellPhone_Phone10,...,PRI_BLT_2024-03-05,PRI_BLT_2022-09-06,PRI_BLT_2020-09-01,PRI_BLT_2020-03-03,PRI_BLT_2018-09-04,PRI_BLT_2016-09-08,PRI_BLT_2016-03-01,PRI_BLT_2014-09-09,PRI_BLT_2012-09-06,PRI_BLT_2012-03-06
0,0,1,LALMA673700606,04PSE0482004,3.512182e+09,(351) 218-1655,351.0,False,2.0,9.782107e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,LALMA160583861,05SHY3075000,NaN,NaN,NaN,NaN,NaN,NaN,...,R,NaN,O,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,3,LALMA162477243,11GJE1731001,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D,NaN,NaN
3,3,4,LALMA161254623,04CBY2900000,4.015240e+09,(401) 524-0148,401.0,True,3.0,NaN,...,NaN,NaN,D,D,D,NaN,NaN,NaN,NaN,NaN
4,4,5,LALMA162631849,02BMT2091001,4.015240e+09,(401) 524-0148,401.0,True,3.0,4.015240e+09,...,NaN,NaN,D,D,D,NaN,R,NaN,NaN,NaN


In [33]:
print(df.dtypes)

Unnamed: 0                   int64
SEQUENCE                     int64
LALVOTERID                  object
Voters_StateVoterID         object
VoterTelephones_Phone10    float64
                            ...   
PRI_BLT_2016-09-08          object
PRI_BLT_2016-03-01          object
PRI_BLT_2014-09-09          object
PRI_BLT_2012-09-06          object
PRI_BLT_2012-03-06          object
Length: 530, dtype: object


In [39]:
df['Precinct'].dropna

<bound method Series.dropna of 0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
           ...    
23302    SALEM 6-1
23303    SALEM 6-1
23304    SALEM 6-1
23305    SALEM 6-1
23306    SALEM 6-1
Name: Precinct, Length: 23307, dtype: object>

In [46]:
# Drop rows where the 'Precinct' column has NaN values
df = df.dropna(subset=['Precinct'])

# Verify that NaN values have been dropped
print(df['Precinct'].isna().sum())  # Should print 0

0


In [47]:
# Define columns for the 2020 general election (example)
votes_2020_columns = [
    'ElectionReturns_G20_Vote_Biden_D', 
    'ElectionReturns_G20_Vote_Trump_R'
]
registered_voters_2020_column = 'ElectionReturns_G20CountyTurnoutAllRegisteredVoters'

# Calculate total votes cast for the 2020 general election
df['Total_Votes_2020'] = df[votes_2020_columns].sum(axis=1)


# Calculate turnout percentage for the 2020 general election
df['Turnout_Percentage_2020'] = (df['Total_Votes_2020'] / df[registered_voters_2020_column]) * 100

# Display the results
print(df[['Precinct', 'Total_Votes_2020', registered_voters_2020_column, 'Turnout_Percentage_2020']].head())

     Precinct  Total_Votes_2020  \
39  SALEM 1-1          412035.0   
40  SALEM 1-1          412035.0   
41  SALEM 1-1          412035.0   
42  SALEM 1-1          412035.0   
43  SALEM 1-1          412035.0   

    ElectionReturns_G20CountyTurnoutAllRegisteredVoters  \
39                                               0.78     
40                                               0.78     
41                                               0.78     
42                                               0.78     
43                                               0.78     

    Turnout_Percentage_2020  
39               52825000.0  
40               52825000.0  
41               52825000.0  
42               52825000.0  
43               52825000.0  


In [61]:
df_t = pd.read_csv('municipal.csv')

In [62]:
df_t['Year'] = pd.to_datetime(df_t['ElectionDate']).dt.year

# Display the updated DataFrame
print(df_t)

               VoterID ElectionDate  TurnoutRate TurnoutPercentage  Year
0       LALMA159116737      9/17/13     0.193267            19.33%  2013
1       LALMA159116737      9/24/13     0.000259             0.03%  2013
2       LALMA159116737      11/5/13     0.359644            35.96%  2013
3       LALMA159116737      9/29/15     0.049954             5.00%  2015
4       LALMA159116737      11/3/15     0.336343            33.63%  2015
...                ...          ...          ...               ...   ...
481405  LALMA702868801      11/2/21     0.400698            40.07%  2021
481406  LALMA702868801      3/28/23     0.239586            23.96%  2023
481407  LALMA702868801      5/16/23     0.304312            30.43%  2023
481408  LALMA702868801      9/19/23     0.019646             1.96%  2023
481409  LALMA702868801      11/7/23     0.209296            20.93%  2023

[481410 rows x 5 columns]


/var/folders/n5/_35mdxvd5tg02wjzpxdfqp0m0000gn/T/ipykernel_28932/826753313.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_t['Year'] = pd.to_datetime(df_t['ElectionDate']).dt.year


In [63]:
df_t.head()

,VoterID,ElectionDate,TurnoutRate,TurnoutPercentage,Year
0,LALMA159116737,9/17/13,0.193267,19.33%,2013
1,LALMA159116737,9/24/13,0.000259,0.03%,2013
2,LALMA159116737,11/5/13,0.359644,35.96%,2013
3,LALMA159116737,9/29/15,0.049954,5.00%,2015
4,LALMA159116737,11/3/15,0.336343,33.63%,2015


In [64]:
df_t = df_t[df_t['Year'].between(2015, 2022)]

In [65]:
# Save the filtered DataFrame to a CSV file
df_t.to_csv('municipal_year.csv', index=False)

In [66]:
df1 = pd.read_csv("municipal_year.csv")

In [67]:
# List of Voter IDs with null values to drop
voter_ids_to_drop = [
    'LALMA673700606', 'LALMA160583861', 'LALMA162477243', 'LALMA161254623', 
    'LALMA162631849', 'LALMA160748892', 'LALMA594641189', 'LALMA581249900', 
    'LALMA162292328', 'LALMA410527984', 'LALMA528337635', 'LALMA162402997', 
    'LALMA471677301', 'LALMA673596352', 'LALMA410341091', 'LALMA461914253', 
    'LALMA159227849', 'LALMA162570090', 'LALMA604136624', 'LALMA493785085', 
    'LALMA161298630', 'LALMA518104853', 'LALMA552636494', 'LALMA474352780', 
    'LALMA583716040', 'LALMA442032686', 'LALMA161269344', 'LALMA581262918', 
    'LALMA641156044', 'LALMA651075659', 'LALMA651075681', 'LALMA673670892', 
    'LALMA162547637', 'LALMA160692551', 'LALMA410155097', 'LALMA159543975', 
    'LALMA410395894', 'LALMA160899592', 'LALMA162395034'
]

# Drop rows with the specified Voter IDs
df_cleaned = df1[~df1['VoterID'].isin(voter_ids_to_drop)]

# Save the cleaned DataFrame to a new CSV file
df_cleaned.to_csv('clean.csv', index=False)

In [68]:
df2 = pd.read_csv("clean.csv")
# List of Voter IDs to check
voter_ids_to_check = [
    'LALMA673700606', 'LALMA160583861', 'LALMA162477243', 'LALMA161254623', 
    'LALMA162631849', 'LALMA160748892', 'LALMA594641189', 'LALMA581249900', 
    'LALMA162292328', 'LALMA410527984', 'LALMA528337635', 'LALMA162402997', 
    'LALMA471677301', 'LALMA673596352', 'LALMA410341091', 'LALMA461914253', 
    'LALMA159227849', 'LALMA162570090', 'LALMA604136624', 'LALMA493785085', 
    'LALMA161298630', 'LALMA518104853', 'LALMA552636494', 'LALMA474352780', 
    'LALMA583716040', 'LALMA442032686', 'LALMA161269344', 'LALMA581262918', 
    'LALMA641156044', 'LALMA651075659', 'LALMA651075681', 'LALMA673670892', 
    'LALMA162547637', 'LALMA160692551', 'LALMA410155097', 'LALMA159543975', 
    'LALMA410395894', 'LALMA160899592', 'LALMA162395034'
]

# Check if each Voter ID exists in the dataset
for voter_id in voter_ids_to_check:
    if voter_id in df2['VoterID'].values:
        print(f"{voter_id} is present in the dataset.")
    else:
        print(f"{voter_id} is NOT present in the dataset.")

LALMA673700606 is NOT present in the dataset.
LALMA160583861 is NOT present in the dataset.
LALMA162477243 is NOT present in the dataset.
LALMA161254623 is NOT present in the dataset.
LALMA162631849 is NOT present in the dataset.
LALMA160748892 is NOT present in the dataset.
LALMA594641189 is NOT present in the dataset.
LALMA581249900 is NOT present in the dataset.
LALMA162292328 is NOT present in the dataset.
LALMA410527984 is NOT present in the dataset.
LALMA528337635 is NOT present in the dataset.
LALMA162402997 is NOT present in the dataset.
LALMA471677301 is NOT present in the dataset.
LALMA673596352 is NOT present in the dataset.
LALMA410341091 is NOT present in the dataset.
LALMA461914253 is NOT present in the dataset.
LALMA159227849 is NOT present in the dataset.
LALMA162570090 is NOT present in the dataset.
LALMA604136624 is NOT present in the dataset.
LALMA493785085 is NOT present in the dataset.
LALMA161298630 is NOT present in the dataset.
LALMA518104853 is NOT present in t

In [69]:
df3 = pd.read_csv('clean.csv')

# Group by VoterID and Year, then calculate the average TurnoutRate
df_avg = df3.groupby(['VoterID', 'Year'], as_index=False)['TurnoutRate'].mean()
df_avg.to_csv('averaged_turnout.csv', index=False)

In [103]:
df4 = pd.read_csv('data.csv')

columns_to_duplicate = ['VoterID', 'Residence_Addresses_Latitude', 'Residence_Addresses_Longitude', 'Precinct']
df_selected = df4[columns_to_duplicate]

df_duplicated = df_selected.loc[df_selected.index.repeat(8)].reset_index(drop=True)

df_duplicated.to_csv('election_data.csv', index=False)

In [114]:
data = pd.read_csv('clustering.csv')
data.isna()

,VoterID,Year,TurnoutRate,Residence_Addresses_Latitude,Residence_Addresses_Longitude,Precinct
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
...,...,...,...,...,...,...
256307,False,False,False,False,False,False
256308,False,False,False,False,False,False
256309,False,False,False,False,False,False
256310,False,False,False,False,False,False


In [30]:
df.dtypes

VoterID                           object
Year                              object
TurnoutRate                      float64
Residence_Addresses_Latitude     float64
Residence_Addresses_Longitude    float64
Precinct                          object
dtype: object

In [22]:
cluster1 = pd.read_csv("clustering.csv", dtype={"Year": str, "Precinct": str})

# Convert TurnoutRate to percentage
cluster1["TurnoutRate"] *= 100

# Aggregate by Year and Precinct to compute the average TurnoutRate
precinct_avg_turnout = cluster1.groupby(["Year", "Precinct"], as_index=False)["TurnoutRate"].mean()

# Sort values for better readability
precinct_avg_turnout = precinct_avg_turnout.sort_values(by=["Year", "TurnoutRate"], ascending=[True, False])
precinct_avg_turnout.style.set_sticky()  

,Year,Precinct,TurnoutRate
0,2015,SALEM 1-1,19.310000
1,2015,SALEM 1-2,19.310000
2,2015,SALEM 2-1,19.310000
3,2015,SALEM 2-2,19.310000
4,2015,SALEM 3-1,19.310000
5,2015,SALEM 3-2,19.310000
6,2015,SALEM 4-1,19.310000
7,2015,SALEM 4-2,19.310000
8,2015,SALEM 5-1,19.310000
9,2015,SALEM 5-2,19.310000


In [26]:
yearly_avg_turnout = cluster1.groupby("Year", as_index=False)["TurnoutRate"].mean()

precinct_avg_turnout = precinct_avg_turnout.sort_values(by=["Year", "TurnoutRate"], ascending=[True, False])
yearly_avg_turnout = yearly_avg_turnout.sort_values(by=["Year"], ascending=True)

print("\nOverall Average Turnout Rate per Year:")
display(yearly_avg_turnout)


Overall Average Turnout Rate per Year:


,Year,TurnoutRate
0,2015,19.31
1,2016,50.40
2,2017,36.00
3,2018,46.51
4,2019,27.75
5,2020,57.23
6,2021,32.63
7,2022,42.79


In [38]:
df = pd.read_csv("clustering.csv", dtype={"Year": str, "Precinct": str})

df["TurnoutRate"] *= 100

precinct_avg = df.groupby("Precinct")["TurnoutRate"].mean().to_dict()

# Function to classify turnout as High or Low
def classify_turnout(row):
    mean_turnout = precinct_avg[row["Precinct"]]
    
    return "High" if row["TurnoutRate"] >= mean_turnout else "Low"

df["TurnoutCategory"] = df.apply(classify_turnout, axis=1)

   Year   Precinct  TurnoutRate TurnoutCategory
0  2015  SALEM 3-2        19.31             Low
1  2016  SALEM 3-2        50.40            High
2  2017  SALEM 3-2        36.00             Low
3  2018  SALEM 3-2        46.51            High
4  2019  SALEM 3-2        27.75             Low


#### Where are the high turnout and low turnout areas located? - using K-means clustering

In [103]:
df = pd.read_csv("clustering.csv", dtype={"Year": str, "Precinct": str})

# Convert TurnoutRate to percentage
df["TurnoutRate"] *= 100

# Compute mean & standard deviation per precinct across years
precinct_stats = df.groupby("Precinct").agg({
    "TurnoutRate": ["mean", "std"]
}).reset_index()
precinct_stats.columns = ["Precinct", "MeanTurnout", "TurnoutSD"]

# Merge stats back into the dataset
df = df.merge(precinct_stats, on="Precinct", how="left")

# Count voters per precinct per year
voter_counts = df.groupby(["Year", "Precinct"]).size().reset_index(name="VoterCount")

# Aggregate TurnoutRate per Precinct per Year
precinct_turnout = df.groupby(["Year", "Precinct"], as_index=False).agg({
    "TurnoutRate": "mean",
    "Residence_Addresses_Latitude": "mean",
    "Residence_Addresses_Longitude": "mean",
    "MeanTurnout": "mean",
    "TurnoutSD": "mean"
})

# Merge with voter counts
precinct_turnout = precinct_turnout.merge(voter_counts, on=["Year", "Precinct"], how="left")

# Function to apply K-Means with 2 clusters
def assign_kmeans_clusters(year):
    year_data = precinct_turnout[precinct_turnout["Year"] == year].copy()
    
    if year_data.empty:
        print(f"No data available for {year}.")
        return year_data

    # Apply K-Means with 2 clusters
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    year_data["Cluster"] = kmeans.fit_predict(year_data[["TurnoutRate"]])
    
    # Map clusters to "Low" and "High" based on centroid values
    cluster_centers = kmeans.cluster_centers_.flatten()
    sorted_indices = np.argsort(cluster_centers)
    category_map = {sorted_indices[0]: "Low", sorted_indices[1]: "High"}
    year_data["TurnoutCategory"] = year_data["Cluster"].map(category_map)
    
    return year_data

def create_turnout_map(year, precinct=None, sd_multiplier=1.0):
    filtered_data = assign_kmeans_clusters(year)
    
    if precinct:
        filtered_data = filtered_data[filtered_data["Precinct"] == precinct]
    
    if filtered_data.empty:
        print(f"No data available for year {year}.")
        return None
    
    # Define a smaller map size
    fig = Figure(width=1100, height=400)  
    m = folium.Map(
        location=[filtered_data["Residence_Addresses_Latitude"].mean(),
                  filtered_data["Residence_Addresses_Longitude"].mean()],
        zoom_start=11,
        width="100%", 
        height="400px" 
    )
    fig.add_child(m) 

    # Dynamic color gradient based on turnout rate
    norm = colors.Normalize(vmin=filtered_data["TurnoutRate"].min(), vmax=filtered_data["TurnoutRate"].max())
    cmap = cm.get_cmap("RdYlGn")
    
    for _, row in filtered_data.iterrows():
        fill_color = colors.to_hex(cmap(norm(row["TurnoutRate"])))

        popup_html = f"""
        <b>Precinct:</b> {row['Precinct']}<br>
        <b>Turnout Rate:</b> {row['TurnoutRate']:.2f}%<br>
        <b>Category:</b> {row['TurnoutCategory']}<br>
        <b>Voters:</b> {row['VoterCount']}<br>
        <b>Mean ± SD:</b> {row['MeanTurnout']:.2f} ± {row['TurnoutSD']:.2f}%
        """
        
        folium.CircleMarker(
            location=(row["Residence_Addresses_Latitude"], row["Residence_Addresses_Longitude"]),
            radius=7,
            color=fill_color,
            fill=True,
            fill_color=fill_color,
            fill_opacity=0.8,
            popup=popup_html
        ).add_to(m)
        
        # Multi-layered uncertainty rings
        for i in range(1, 3):
            folium.Circle(
                location=(row["Residence_Addresses_Latitude"], row["Residence_Addresses_Longitude"]),
                radius=row["TurnoutSD"] * sd_multiplier * i * 10,
                color=fill_color,
                fill=True,
                fill_opacity=0.2 / i
            ).add_to(m)
    
    # 🔹 **Add Color Legend**
    min_turnout, max_turnout = filtered_data["TurnoutRate"].min(), filtered_data["TurnoutRate"].max()
    
    legend_html = f"""
    <div style="position: fixed; 
                bottom: 50px; left: 50px; width: 180px; height: 120px; 
                background-color: white; z-index:9999; font-size:12px;
                border:2px solid grey; padding: 10px;
                border-radius: 5px;">
        <b>Turnout Categories</b><br>
        <div style="display: flex; align-items: center;">
            <div style="background: red; width: 15px; height: 15px; margin-right: 5px;"></div>
            <span>Low Turnout</span>
        </div>
        <div style="display: flex; align-items: center;">
            <div style="background: green; width: 15px; height: 15px; margin-right: 5px;"></div>
            <span>High Turnout</span>
        </div>
        <br><b>Turnout Rate Scale</b><br>
        <span style="background: linear-gradient(to right, red, yellow, green); 
                     width: 100%; display: inline-block; height: 10px;"></span><br>
        <span style="display: flex; justify-content: space-between;">
        </span>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    return m

# Widgets for user selection
year_dropdown = widgets.Dropdown(
    options=sorted(precinct_turnout["Year"].unique()),
    description='Year:'
)

precinct_dropdown = widgets.Dropdown(
    options=['All'] + sorted(precinct_turnout["Precinct"].unique()),
    description='Precinct:'
)

sd_slider = widgets.FloatSlider(
    value=1.0, min=0.5, max=3.0, step=0.5,
    description='SD Multiplier:'
)

output = widgets.Output()

def update_map(year, precinct, sd_multiplier):
    with output:
        clear_output(wait=True)
        selected_precinct = None if precinct == 'All' else precinct
        turnout_map = create_turnout_map(year, selected_precinct, sd_multiplier)
        if turnout_map:
            display(turnout_map)

interactive_widget = widgets.interactive(
    update_map, 
    year=year_dropdown,
    precinct=precinct_dropdown,
    sd_multiplier=sd_slider
)

display(widgets.VBox([year_dropdown, precinct_dropdown, sd_slider, output]))
